# Thai Air Intelligence — PM2.5 Regression + Classification

Trains six regression families and six matching classifiers independently per province. The classification target is derived from the **actual next-day PM2.5 target**, never from a regression prediction.

Required Colab Secrets: `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY`.

Safety defaults: `REGISTER=False`, `ACTIVATE=False`. Existing production models remain unchanged until both flags are explicitly reviewed.

In [ ]:
# 1. Configuration
REGISTER = False
ACTIVATE = False
PROVINCE = "all"  # e.g. TH-30
MINIMUM_ROWS = 180
MODEL_FAMILIES = [
    "random_forest", "adaboost", "gradient_boosting",
    "xgboost", "lightgbm", "catboost",
]
SERVING_POLICY = "classifier_with_regression_fallback"

if ACTIVATE and not REGISTER:
    raise ValueError("ACTIVATE=True requires REGISTER=True")


In [ ]:
# 2. Fetch the reviewed feature branch and install pinned dependencies
!rm -rf /content/THAI-AIR-INTELLIGENCE-LITE
!git clone --depth 1 --branch feature/regression-classification-pm25 https://github.com/kzabCde/THAI-AIR-INTELLIGENCE-LITE.git /content/THAI-AIR-INTELLIGENCE-LITE
%cd /content/THAI-AIR-INTELLIGENCE-LITE
%pip install -q -r training/requirements.txt


In [ ]:
# 3. Load server-side Supabase secrets without printing values
import os
from google.colab import userdata

for name in ("SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY"):
    value = userdata.get(name)
    if not value:
        raise ValueError(f"Missing Colab Secret: {name}")
    os.environ[name] = value
print("Supabase secrets loaded")


In [ ]:
# 4. Verify shared PM2.5 class boundaries
from training.pm25_classes import class_for_pm25, THRESHOLD_VERSION

boundary_cases = {0: 1, 15: 1, 15.01: 2, 25: 2, 25.01: 3,
                  37.5: 3, 37.51: 4, 75: 4, 75.01: 5}
assert all(class_for_pm25(value) == expected for value, expected in boundary_cases.items())
print("Thresholds verified:", THRESHOLD_VERSION)


In [ ]:
# 5. Run the leakage-safe dual training pipeline
import subprocess
import sys

args = [
    sys.executable, "-m", "training.train_dual_models",
    "--min-rows", str(MINIMUM_ROWS),
    "--serving-policy", SERVING_POLICY,
]
if not REGISTER:
    args.append("--dry-run")
else:
    args.append("--register")
if ACTIVATE:
    args.append("--activate")
if PROVINCE != "all":
    args.extend(["--province", PROVINCE])
for family in MODEL_FAMILIES:
    args.extend(["--model-family", family])

completed = subprocess.run(args, check=False)
if completed.returncode != 0:
    raise RuntimeError(f"Training exited with code {completed.returncode}")


In [ ]:
# 6. Display the structured run summary
import json
from pathlib import Path
import pandas as pd

summary_path = max(Path("training/artifacts").glob("*/run_summary.json"), key=lambda p: p.stat().st_mtime)
summary = json.loads(summary_path.read_text(encoding="utf-8"))
display(pd.DataFrame(summary["results"]))
print({key: summary[key] for key in ("run_id", "register", "activate", "successful_provinces", "failed_provinces")})
